# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [11]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display



In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [4]:
# here is the question; type over this to ask something new
system_prompt = """
You are my handy dandy tech advisor and your job is to explain me the technical questions I ask you.
You need to be friendly, but not too much. Explain it the most proefficient way that you can.
"""
user_prompt_prefix = """
Here is my question.

"""
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

def messages_for(question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + question}
    ]

In [5]:
# Get gpt-4o-mini to answer, with streaming
openai = OpenAI()

def stream(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages_for(question),
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


stream(question)


This code snippet is using Python's generator capabilities along with a set comprehension to yield author names from a collection of `books`. Let’s break it down step by step:

1. **Set Comprehension**: 
   - The expression `{book.get("author") for book in books if book.get("author")}` is a set comprehension. It iterates over each `book` in the `books` iterable.
   - `book.get("author")` retrieves the value associated with the key `"author"` from each `book` dictionary. 
   - The condition `if book.get("author")` ensures that only books with a valid (non-None, non-empty) author entry are included in the set. 

2. **Set Behavior**: 
   - The result of the set comprehension is a set of unique authors, since sets automatically handle duplicates by only storing unique entries.

3. **Yielding Values**:
   - The keyword `yield from` is used to yield each value from the set generated by the set comprehension. This means the code will output each author one by one, allowing for iteration over the resulting set as if it were a generator.

### Why Use This?

- **Efficiency**: Using a set comprehension ensures that the list of authors contains no duplicates without additional logic. 
- **Memory Management**: By using `yield from`, it produces results one at a time instead of creating a full list in memory, which can be more memory efficient especially with large data sets.
- **Conciseness**: It succinctly combines the logic of filtering (removing books without an author) and collecting (only unique authors).

### Summary

In summary, this code snippet efficiently collects unique authors from a list of book dictionaries and yields them one at a time. It utilizes both set comprehension for uniqueness and generator capabilities for memory-efficient iteration.

In [8]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
# Get Llama 3.2 to answer
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

response = ollama.chat.completions.create(
    model = MODEL_LLAMA,
    messages = messages_for(question)
)
result = response.choices[0].message.content
display(Markdown(result))

Let's break down this line of code step by step.

**What the code does:**

This line of code uses a feature called "generator expression" to generate an iterable sequence of values. It's similar to a list comprehension, but instead of building a new list in memory, it generates values on-the-fly as they're needed.

Here's what this specific code does:

1. `for book in books`: Iterate over each book object in the `books` collection.
2. `if book.get("author")`: Filter out any books that don't have an "author" key or value associated with them (this is equivalent to `book["author"] != None`, but using Python's dictionary shortcut syntax).
3. `{ book.get("author") for book in books if book.get("author") }`: For each book that passes the filter, get the author name from the dictionary and yield it as a value.
4. `yield from ...`: This line sends all the yielded values to an external context (in this case, likely an iterable) using the `yield from` syntax. The `yield from` statement is short for "pass through each yielded value".

So overall, this code generates an iterable sequence of author names, but only includes authors from books that have an author associated with them.

**Why:**

In general, generator expressions like this one are useful when you need to process large datasets or collections without having to load everything into memory at once. By yielding values as needed, the function can efficiently generate data on-the-fly rather than storing it all in memory.

This specific code might be used in situations where:

* You want to iterate over a large list of books but only care about the authors.
* You need to filter out invalid or incomplete book data based on the "author" key.
* You're working with an API or data source that returns responses as iterators, and this kind of expression helps you extract useful data from it.

Example use case:
```python
books = [
  {"title": "Book 1", "author": "John Doe"},
  {"title": "Book 2", "author": None},
  {"title": "Book 3", "author": "Jane Smith"}
]

for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```
Output:
```
John Doe
Jane Smith
```